# ViterbiNet Transformer - Google Colab Training

This notebook runs the ViterbiNet channel estimation and equalization training pipeline on Google Colab GPU.

**Requirements:**
- GPU Runtime (Runtime > Change runtime type > GPU)
- Upload your code files or connect to Google Drive

**Note (this revision):** clones the `claude/transformer-sionna-mlp-comparison-wc67zp` branch, which fixes a regression in `ECC_Transformer.generate_square_subsequent_mask` (Code/models.py) that was silently disabling the Transformer's self-attention -- see README.md section 5.3. Cells 5/8/9 (config, cache pre-gen, training loop) were also updated to match the self-supervised methodology used for `Results/metrics/model_performance_final_mc_83.csv`, so results here are directly comparable to that baseline, and a bug where the cache pre-generation used a different `memory_length` than training (so was silently never reused) is fixed.


## 1. Setup: Mount Google Drive (Optional)

In [1]:
# Uncomment to mount Google Drive if your code is stored there
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/viterbitransformed

Mounted at /content/drive
/content/drive/MyDrive/viterbitransformed


## 2. Clone Repository from GitHub

In [ ]:
# Clone the repo, on the branch with the attention-mask fix applied
# (Code/models.py: ECC_Transformer.generate_square_subsequent_mask was restoring
# the wrong -- all-True -- mask, which silently disabled self-attention).
import os

REPO_DIR = "viterbitransformed"
BRANCH = "claude/transformer-sionna-mlp-comparison-wc67zp"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/Gilzuk/viterbitransformed.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {REPO_DIR}
!git log --oneline -3


## 3. Install Dependencies

In [3]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy scipy matplotlib pandas psutil tqdm

## 4. Verify GPU Availability

In [4]:
import torch
import os

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{'='*60}")
if device.type == "cuda":
    print(f"GPU DETECTED: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Compute Capability: {torch.cuda.get_device_capability()[0]}.{torch.cuda.get_device_capability()[1]}")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")
print(f"{'='*60}")

# Set memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

GPU DETECTED: Tesla T4
GPU Memory: 14.74 GB
CUDA Version: 12.6
Compute Capability: 7.5


## 5. Configuration Parameters

Adjust these parameters based on your Colab GPU (typically 15GB for free tier):

In [ ]:
# Configuration for Colab.
#
# This mirrors the self-supervised online-training setup used to produce
# Results/metrics/model_performance_final_mc_83.csv (the pre-fix baseline
# documented in README.md section 5.3) so the results here are directly
# comparable, just at full scale on GPU instead of this session's reduced
# CPU-only sanity check (SNR 4/12/16, n=3).
CONFIG = {
    'val_frames': 12,
    'train_frames': 12,
    'subframes_in_frame': 25,
    'train_minibatch_num': 25,

    # Full SNR ladder (0-17 dB), constant block length/pilots -- matches mc_83.csv.
    'parameters': [(snr, 120, 25) for snr in range(0, 18)],
    'models_list': ['Transformer', 'ViterbiNet'],

    # num_iterations = independent Monte-Carlo restarts (this is what gives you
    # a mean +/- CI per SNR point, same as the "n" in README 5.3). Start small
    # to confirm everything runs, then raise it -- metrics are saved after every
    # iteration, so it's safe to stop early and resume by re-running this cell
    # with run_over=1 (load trained weights) instead of 2.
    #   3  -> ~18 SNR * 2 models * 3   =  108 runs
    #   10 -> ~18 SNR * 2 models * 10  =  360 runs (full-scale reproduction)
    'num_iterations': 3,
    'num_reps': [1] * 18,   # evaluation repetitions per (snr, model) call within one iteration

    'run_over': 2,             # 0=load plots, 1=load weights, 2=train from scratch
    'self_supervised': True,   # matches the paper's online-training setup and mc_83.csv
}

print("Configuration:")
print(f"  Batch size: {CONFIG['val_frames'] * CONFIG['subframes_in_frame']}")
print(f"  Models: {CONFIG['models_list']}")
print(f"  SNR values: {[p[0] for p in CONFIG['parameters']]}")
print(f"  Iterations: {CONFIG['num_iterations']}  ->  {len(CONFIG['parameters']) * len(CONFIG['models_list']) * CONFIG['num_iterations']} total runs")


## 6. Import Modules

In [23]:
from Code.dir_definitions import *
from Code.plotter import get_ser_data
from Code.trainer import Trainer
from Code.csv_reporter import ModelPerformanceTracker
from Code.channel.data_cache import ChannelDataCache
import torch
import gc
import numpy as np
import time
from datetime import datetime
import psutil

print("✓ All modules imported successfully")

✓ All modules imported successfully


## 7. Pre-generate Cache Data

This step generates and caches channel data to reduce CPU load during training:

In [ ]:
# Setup hyperparameters
HYPERPARAMS_DICT = {
    'noisy_est_var': 0,
    'memory_length': 4,   # must match Trainer's default (configuration.yaml) or
                           # the pre-generated cache below is never reused
    'fading_taps_type': 1,
    'fading_in_channel': False,   # cost2100 -> fading_in_channel False (see main.py)
    'fading_in_decoder': True,
    'gamma': 0.2,
    'channel_type': 'ISI_AWGN',
    'train_frames': CONFIG['train_frames'],
    'val_frames': CONFIG['val_frames'],
    'subframes_in_frame': CONFIG['subframes_in_frame'],
    'self_supervised_iterations': 200,
    'ser_thresh': 0.02,
    'train_minibatch_num': CONFIG['train_minibatch_num'],
    'n_symbols': 2,
    'channel_coefficients': 'cost2100',
}

# Pre-generate cache (comment out if already generated)
from Code.channel.channel_dataset import ChannelModelDataset
from numpy.random import RandomState

print("Pre-generating cache data...")
cache = ChannelDataCache()

for snr, val_block_length, pilot_length in CONFIG['parameters']:
    for phase in ['train', 'val']:
        print(f"  Generating SNR={snr}, block_length={val_block_length}, phase={phase}")
        transmission_length = val_block_length + 8 * HYPERPARAMS_DICT['n_symbols']

        dataset = ChannelModelDataset(
            channel_type=HYPERPARAMS_DICT['channel_type'],
            block_length=val_block_length,
            transmission_length=transmission_length,
            words=HYPERPARAMS_DICT['val_frames'] * HYPERPARAMS_DICT['subframes_in_frame'],
            memory_length=HYPERPARAMS_DICT['memory_length'],
            channel_coefficients=HYPERPARAMS_DICT['channel_coefficients'],
            random=RandomState(),
            word_rand_gen=RandomState(),
            noisy_est_var=HYPERPARAMS_DICT['noisy_est_var'],
            fading_taps_type=HYPERPARAMS_DICT['fading_taps_type'],
            use_ecc=True,
            n_symbols=HYPERPARAMS_DICT['n_symbols'],
            fading_in_channel=HYPERPARAMS_DICT['fading_in_channel'],
            fading_in_decoder=HYPERPARAMS_DICT['fading_in_decoder'],
            phase=phase
        )
        _ = dataset.__getitem__([snr], HYPERPARAMS_DICT['gamma'])
        del dataset
        gc.collect()

print("Cache generation complete!")
cache.get_cache_stats()


## 8. Training Loop

In [ ]:
# Initialize performance tracker
from Code.complexity import profile_model

perf_tracker = ModelPerformanceTracker()

# Enable CUDA optimizations
if device.type == "cuda":
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = True
    torch.backends.cuda.matmul.allow_fp16_accumulation = True

# Main training loop
for mc in range(CONFIG['num_iterations']):
    print(f"\n{'='*70}")
    print(f"ITERATION {mc + 1}/{CONFIG['num_iterations']}")
    print(f"{'='*70}")

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    for idx, (snr, val_block_length, pilot_length) in enumerate(CONFIG['parameters'], 1):
        print(f'\n{"-"*70}')
        print(f'Parameter Set [{idx}/{len(CONFIG["parameters"])}]:')
        print(f'   SNR={snr}, val_block_length={val_block_length}, pilots={pilot_length}')
        print(f'{"-"*70}')

        # Update hyperparameters
        HYPERPARAMS_DICT['curr_SNR'] = snr
        HYPERPARAMS_DICT['val_block_length'] = val_block_length
        HYPERPARAMS_DICT['train_block_length'] = val_block_length
        HYPERPARAMS_DICT['pilots_num'] = pilot_length

        current_params = f"{HYPERPARAMS_DICT['channel_coefficients']}_{snr}_{val_block_length}_{HYPERPARAMS_DICT['n_symbols']}_mc{mc}"

        for model_idx, model_name in enumerate(CONFIG['models_list'], 1):
            print(f"\n[{model_idx}/{len(CONFIG['models_list'])}] Processing model: {model_name}")

            # Create trainer
            method_name = f"{model_name}_ModelBased"
            if 'trainer' in locals() or 'trainer' in globals():
              del trainer
            trainer = Trainer(
                model_name=model_name,
                detector_method='ModelBased',
                self_supervised=CONFIG['self_supervised'],
                weights_dir=os.path.join(WEIGHTS_DIR, f'{method_name}_training_{val_block_length}_{HYPERPARAMS_DICT["n_symbols"]}_channel1_{HYPERPARAMS_DICT["channel_coefficients"]}_mc{mc}'),
                **HYPERPARAMS_DICT
            )

            # Move model to GPU, and get its real parameter count (was hardcoded 0)
            model_size = 0
            if hasattr(trainer.detector, "model"):
                trainer.detector.model.to(device)
                print(f"Model loaded on {device}")
                model_size = sum(p.numel() for p in trainer.detector.model.parameters() if p.requires_grad)

            complexity = None
            if hasattr(trainer.detector, "model"):
                try:
                    complexity = profile_model(
                        trainer.detector.model,
                        transmission_length=val_block_length + 8 * HYPERPARAMS_DICT['n_symbols'],
                        memory_length=HYPERPARAMS_DICT['memory_length'],
                        device=device)
                except Exception as e:
                    print(f"complexity profiling skipped for {model_name}: {e}")

            # Train and evaluate
            perf_tracker.start_timing(model_name)

            ser = get_ser_data(
                trainer,
                run_over=CONFIG['run_over'],
                num_of_rep=CONFIG['num_reps'][idx-1],
                method_name=f"{method_name}_{current_params}",
                device=device,
                dtype=torch.float32
            )

            run_time = perf_tracker.end_timing(model_name)
            final_ser = float(np.mean(ser))

            print(f"\nResults: SER={final_ser:.6f}, Time={run_time:.2f}s, params={model_size}")

            # Save metrics (ser_reps lets record_metrics compute a 95% CI)
            ser_reps = np.asarray(ser).reshape(-1).tolist()
            perf_tracker.record_metrics(
                model_name=model_name,
                snr=snr,
                final_ser=final_ser,
                model_size=model_size,
                run_time=run_time,
                config={
                    'memory_length': HYPERPARAMS_DICT['memory_length'],
                    'noisy_est_var': HYPERPARAMS_DICT['noisy_est_var'],
                    'channel_coefficients': HYPERPARAMS_DICT['channel_coefficients'],
                    'n_symbols': HYPERPARAMS_DICT['n_symbols'],
                    'val_block_length': val_block_length,
                    'pilots_num': pilot_length,
                    'detector_method': 'ModelBased',
                },
                complexity=complexity,
                ser_reps=ser_reps,
            )

            # Cleanup
            del trainer
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

    # Save metrics after each iteration
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_file = f"colab_metrics_iter_{mc}_{timestamp}.csv"
    perf_tracker.save_to_csv(csv_file)
    print(f"\nMetrics saved to: {csv_file}")

print("\n" + "="*70)
print("ALL TRAINING COMPLETED!")
print("="*70)


## 8b. Summary tables (BER-vs-SNR pivot + complexity)


In [ ]:
# Pivot into the model-vs-SNR table used in README.md 5.1/5.3, and the
# BER-gain-vs-complexity table the paper reviewers asked for.
ber_table = perf_tracker.save_ber_vs_snr_table()
complexity_table = perf_tracker.save_complexity_table()


In [40]:
if 'trainer' in locals() or 'trainer' in globals():
  del trainer

In [13]:
import matplotlib.pyplot as plt
import numpy as np
from Code.dir_definitions import PLOTS_DIR
import os
import pickle
import math

def save_pkl(path, obj):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def get_ser_data(trainer, run_over, num_of_rep, method_name, device, dtype, use_amp=False, scaler=None):
    plots_dir = os.path.join(PLOTS_DIR, 'ser')
    plots_path = os.path.join(plots_dir, method_name + '.pkl')

    if not os.path.exists(plots_dir):
        os.makedirs(plots_dir)

    if run_over == 0 and os.path.exists(plots_path):
        print(f"loading plots from {plots_path}")
        ser_total = load_pkl(plots_path)

    else:
        print("calculating fresh")
        try:
            ser_total = trainer.run(run_over, num_of_rep=num_of_rep,
                                   device_arg=device, dtype=dtype, use_amp=use_amp, scaler=scaler)
        except TypeError:
            # Fallback if trainer.run doesn't support these parameters
            ser_total = trainer.run(run_over, num_of_rep=num_of_rep)
        save_pkl(plots_path, ser_total)
    print(np.mean(ser_total))
    return ser_total

def plot_ser_curves(ser_data, snr_range, title="SER vs. SNR", filename="ser_curve.png"):
    plt.figure()
    for label, ser_values in ser_data.items():
        plt.semilogy(snr_range, ser_values, 'o-', label=label)

    plt.xlabel('SNR (dB)')
    plt.ylabel('SER')
    plt.title(title)
    plt.grid(True, which="both", ls="-")
    plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, filename))
    plt.show()

def plot_ser_curves_different_pilots(ser_data_dict, title="SER vs. SNR for different pilot lengths", filename="ser_curve_pilots.png"):
    plt.figure()
    markers = ['o-', 's--', '^-', 'x-'] # Different markers for different pilot lengths
    for i, (pilot_length, ser_data) in enumerate(ser_data_dict.items()):
        marker = markers[i % len(markers)]
        for label, ser_values in ser_data.items():
            plt.semilogy(ser_data['snr_range'], ser_values, marker, label=f'{label} (Pilots: {pilot_length})')

    plt.xlabel('SNR (dB)')
    plt.ylabel('SER')
    plt.title(title)
    plt.grid(True, which="both", ls="-")
    plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, filename))
    plt.show()

def plot_constellation(channel_output, detected_symbols, title='Constellation Diagram', filename='constellation.png'):
    plt.figure(figsize=(8, 8))
    plt.scatter(channel_output.real, channel_output.imag, alpha=0.5, label='Channel Output')
    plt.scatter(detected_symbols.real, detected_symbols.imag, alpha=0.5, label='Detected Symbols')
    plt.title(title)
    plt.xlabel('In-phase')
    plt.ylabel('Quadrature')
    plt.grid(True)
    plt.axhline(0, color='black',linewidth=0.5)
    plt.axvline(0, color='black',linewidth=0.5)
    plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, filename))
    plt.show()

def visualize_losses(losses, filename='losses.png'):
    plt.figure()
    plt.plot(losses)
    plt.title('Training Loss over Minibatches')
    plt.xlabel('Minibatch Number')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.savefig(os.path.join(PLOTS_DIR, filename))
    plt.show()

def plot_channel_response(h, title='Channel Impulse Response', filename='channel_response.png'):
    plt.figure()
    plt.stem(np.abs(h))
    plt.title(title)
    plt.xlabel('Tap')
    plt.ylabel('Magnitude')
    plt.grid(True)
    plt.savefig(os.path.join(PLOTS_DIR, filename))
    plt.show()

def plot_training_and_validation_loss(training_losses, validation_losses, filename='training_validation_losses.png'):
    epochs = range(1, len(training_losses) + 1)
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, training_losses, 'b', label='Training Loss')
    plt.plot(epochs, validation_losses, 'r', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(PLOTS_DIR, filename))
    plt.show()


## 9. Download Results

Download the trained weights and metrics:

In [ ]:
# List generated files
!ls -lh *.csv

# Download files (uncomment to download)
# from google.colab import files
# files.download('colab_metrics_iter_0_*.csv')

# Or copy to Google Drive if mounted
# !cp *.csv /content/drive/MyDrive/viterbitransformed/Results/